# Fraud Risk Agent → Pega (A2A + OAuth 2.0), hosted with ngrok

This notebook runs the Fraud Risk Agent, gives it a public **HTTPS** address with ngrok, and creates the
OAuth 2.0 **client credentials** that Pega needs. At the end it shows:

| Pega needs | You get |
|---|---|
| Client ID | `pega-claim-assist` |
| Client Secret | a long random secret |
| Access Token Endpoint | `https://<your-ngrok-address>/oauth/token` |
| Scope | `fraud.assess` |
| (and) Agent Card URL | `https://<your-ngrok-address>/.well-known/agent.json` |

**Before you start (once):**
1. Create a free ngrok account and copy your **authtoken** from https://dashboard.ngrok.com/get-started/your-authtoken
   - **Colab:** click the key icon (Secrets) on the left, add a secret named `NGROK_AUTHTOKEN` with the token, and switch on notebook access.
   - **Jupyter:** you will be asked for it (the input is hidden).
2. Have **`fraud-risk-agent.zip`** (the latest version, with OAuth support).
   - **Colab:** you will be asked to upload it.
   - **Jupyter:** put the zip next to this notebook, or open this notebook from inside the `fraud-risk-agent` folder.

**Then:** *Runtime → Run all* (Colab) or *Run → Run All Cells* (Jupyter). Keep the notebook running while Pega uses the agent.

In [ ]:
# 1. Settings: the defaults work; change them only if you need to
CLIENT_ID = "pega-claim-assist"   # the Client ID you enter in Pega
CLIENT_SECRET = ""                # empty: a random secret is made for you. To keep the one already saved in Pega
                                  # after a restart, paste it here (letters, digits, - and _ only)
SCOPE = "fraud.assess"            # the scope Pega asks for
NGROK_DOMAIN = ""                 # optional: your free static ngrok domain, e.g. "your-name.ngrok-free.app",
                                  # so the address stays the same every time (ngrok dashboard -> Domains)
PORT = 8000                       # local port; change it if 8000 is already in use on this computer
USE_NGROK = True                  # False = reachable from this computer only (no Pega), for a quick test

In [ ]:
# 2. Find the agent files (in Colab: upload fraud-risk-agent.zip when asked)
import pathlib
import sys
import zipfile

IN_COLAB = "google.colab" in sys.modules


def find_agent_dir():
    here = pathlib.Path.cwd()
    for folder in [here, *sorted(here.glob("*")), *sorted(here.glob("*/*"))]:
        if (folder / "server.py").is_file() and (folder / "fra_agent.py").is_file():
            return folder.resolve()
    return None


def is_agent_zip(path):
    try:
        with zipfile.ZipFile(path) as z:
            return any(name.endswith("fra_agent.py") for name in z.namelist())
    except zipfile.BadZipFile:
        return False


def unzip_agent(path):
    with zipfile.ZipFile(path) as z:  # skip a Windows virtual environment, caches and any .env file
        members = [n for n in z.namelist()
                   if "/.venv/" not in f"/{n}" and "__pycache__" not in n and not n.endswith(".env")]
        z.extractall(pathlib.Path.cwd(), members=members)


AGENT_DIR = find_agent_dir()
if AGENT_DIR is None:
    zips = [p for p in sorted(pathlib.Path.cwd().glob("*.zip")) if is_agent_zip(p)]
    if not zips and IN_COLAB:
        from google.colab import files
        print("Choose fraud-risk-agent.zip to upload:")
        zips = [p for p in map(pathlib.Path, files.upload()) if is_agent_zip(p)]
    for path in zips:
        unzip_agent(path)
    AGENT_DIR = find_agent_dir()
if AGENT_DIR is None:
    raise FileNotFoundError("The agent files were not found. Put fraud-risk-agent.zip next to this notebook "
                            "(in Colab: upload it when asked) and run this cell again.")
if "FRA_OAUTH_ALLOW_HTTP" not in (AGENT_DIR / "server.py").read_text(encoding="utf-8"):
    raise RuntimeError(f"{AGENT_DIR / 'server.py'} is an older version without OAuth support. "
                       "Use the latest fraud-risk-agent.zip, then run this cell again.")
print("Agent files:", AGENT_DIR)

In [ ]:
# 3. Install the packages (about a minute the first time)
import subprocess

result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(AGENT_DIR / "requirements-server.txt"),
                         "pyngrok>=7.2,<9"], capture_output=True, text=True)
if result.returncode != 0:
    print(result.stdout[-3000:], result.stderr[-3000:])
    raise RuntimeError("Installing the packages failed (details above).")

import jwt  # noqa: E402  (PyJWT, imported after the install on purpose)

if tuple(int(x) for x in jwt.__version__.split(".")[:2]) < (2, 13):
    raise RuntimeError("An older PyJWT is still loaded. Restart the runtime/kernel, then run all cells again.")
print("Packages ready")

In [ ]:
# 4. The client credentials for Pega, and the key that signs access tokens
import json
import secrets

from cryptography.hazmat.primitives.asymmetric import rsa

# Kept when you run all cells again in the same session, so Pega's settings stay valid.
CLIENT_SECRET = CLIENT_SECRET or globals().get("_GENERATED_SECRET") or secrets.token_urlsafe(32)
_GENERATED_SECRET = CLIENT_SECRET
SIGNING_KEY = globals().get("SIGNING_KEY") or rsa.generate_private_key(public_exponent=65537, key_size=2048)
KEY_ID = globals().get("KEY_ID") or secrets.token_hex(8)
PUBLIC_JWK = {**json.loads(jwt.algorithms.RSAAlgorithm.to_jwk(SIGNING_KEY.public_key())),
              "kid": KEY_ID, "use": "sig", "alg": "RS256"}
print("Client ID:", CLIENT_ID, "| Client Secret: ready (shown in step 8)")

In [ ]:
# 5. Give the agent a public HTTPS address with ngrok
import getpass
import os


def ngrok_authtoken():
    token = os.environ.get("NGROK_AUTHTOKEN", "")
    if not token and IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("NGROK_AUTHTOKEN") or ""
        except Exception:  # no such secret, or notebook access not switched on
            token = ""
    return token.strip() or getpass.getpass("Paste your ngrok authtoken (input hidden): ").strip()


if USE_NGROK:
    from pyngrok import conf, ngrok

    ngrok.kill()  # close the tunnel of an earlier run
    conf.get_default().auth_token = ngrok_authtoken()
    try:
        tunnel = ngrok.connect(f"127.0.0.1:{PORT}", "http", **({"domain": NGROK_DOMAIN} if NGROK_DOMAIN else {}))
    except Exception as exc:
        raise RuntimeError(f"ngrok could not start: {exc}\nCheck your authtoken (and NGROK_DOMAIN if you set it). "
                           "A free account allows one ngrok session at a time: stop any other one first.") from None
    PUBLIC_URL = tunnel.public_url.replace("http://", "https://", 1)
else:
    PUBLIC_URL = f"http://127.0.0.1:{PORT}"
    print("USE_NGROK is False: Pega cannot reach this address.")
print("Public address:", PUBLIC_URL)

In [ ]:
# 6. Start the agent, with a small OAuth 2.0 token service for Pega
import base64
import importlib
import socket
import threading
import time
import urllib.parse

from collections import deque

import uvicorn
from fastapi import Request
from fastapi.responses import JSONResponse, Response

TOKEN_URL = f"{PUBLIC_URL}/oauth/token"
AUDIENCE = "fraud-risk-agent"  # every token says it is meant for this agent
TOKEN_LIFETIME = 3600          # seconds; Pega fetches a new token when one expires

# The agent's own OAuth settings (see DEPLOYMENT.md section 4). It checks every token like it would one from
# Keycloak or Entra ID: signature (with the public key below), issuer, audience, expiry and scope.
os.environ.pop("FRA_API_TOKEN", None)
os.environ.update({
    "FRA_PUBLIC_URL": PUBLIC_URL,
    "FRA_OAUTH_ISSUER": PUBLIC_URL,
    "FRA_OAUTH_AUDIENCE": AUDIENCE,
    "FRA_OAUTH_SCOPE": SCOPE,
    "FRA_OAUTH_TOKEN_URL": TOKEN_URL,
    # The agent reads the token service's public key inside this machine, so plain http is safe here.
    "FRA_OAUTH_JWKS_URL": f"http://127.0.0.1:{PORT}/oauth/jwks",
    "FRA_OAUTH_ALLOW_HTTP": "1",
    "FRA_USE_LLM": "0",  # template explanations: fast and identical every time
})
for name in ("NO_PROXY", "no_proxy"):  # never send that local call through a proxy
    os.environ[name] = ",".join(filter(None, ["127.0.0.1", "localhost", os.environ.get(name, "")]))

if str(AGENT_DIR) not in sys.path:
    sys.path.insert(0, str(AGENT_DIR))
old = sys.modules.get("server")
if old is not None and pathlib.Path(getattr(old, "__file__", "")).parent == AGENT_DIR:
    server = importlib.reload(old)  # a fresh app that reads the settings above
else:
    sys.modules.pop("server", None)
    server = importlib.import_module("server")


def oauth_error(status, error):
    headers = {"Cache-Control": "no-store"}
    if status == 401:
        headers["WWW-Authenticate"] = 'Basic realm="fraud-risk-agent"'
    return JSONResponse({"error": error}, status_code=status, headers=headers)


async def token_endpoint(request: Request):
    """OAuth 2.0 client credentials grant (RFC 6749 s4.4): Pega swaps its Client ID and Secret for a token."""
    form = urllib.parse.parse_qs((await request.body()).decode("utf-8", "replace"))
    client_id, client_secret = form.get("client_id", [""])[0], form.get("client_secret", [""])[0]
    auth = request.headers.get("authorization", "")
    if auth[:6].lower() == "basic ":  # credentials in the header ("Basic"); otherwise they came in the form
        try:
            client_id, _, client_secret = base64.b64decode(auth[6:]).decode("utf-8").partition(":")
        except ValueError:
            return oauth_error(401, "invalid_client")
    if form.get("grant_type", [""])[0] != "client_credentials":
        return oauth_error(400, "unsupported_grant_type")
    if not (secrets.compare_digest(client_id.encode(), CLIENT_ID.encode())
            & secrets.compare_digest(client_secret.encode(), CLIENT_SECRET.encode())):
        return oauth_error(401, "invalid_client")
    now = int(time.time())
    claims = {"iss": PUBLIC_URL, "sub": CLIENT_ID, "azp": CLIENT_ID, "aud": AUDIENCE, "scope": SCOPE,
              "iat": now, "exp": now + TOKEN_LIFETIME, "jti": secrets.token_hex(8)}
    token = jwt.encode(claims, SIGNING_KEY, algorithm="RS256", headers={"kid": KEY_ID})
    return JSONResponse({"access_token": token, "token_type": "Bearer", "expires_in": TOKEN_LIFETIME, "scope": SCOPE},
                        headers={"Cache-Control": "no-store", "Pragma": "no-cache"})


async def jwks_endpoint():
    return {"keys": [PUBLIC_JWK]}  # public key only


server.app.add_api_route("/oauth/token", token_endpoint, methods=["POST"], include_in_schema=False)
server.app.add_api_route("/oauth/jwks", jwks_endpoint, methods=["GET"], include_in_schema=False)

# Keep the last 10 calls in memory (never on disk), so step 9 can show exactly what Pega sent.
RECENT_CALLS = deque(maxlen=10)


@server.app.middleware("http")
async def remember_calls(request: Request, call_next):
    if request.url.path not in ("/a2a", "/v1/assess"):
        return await call_next(request)
    sent = await request.body()
    response = await call_next(request)
    answer = b"".join([chunk async for chunk in response.body_iterator])
    RECENT_CALLS.append({"time": time.strftime("%H:%M:%S"), "path": request.url.path,
                         "status": response.status_code, "sent": sent[:20000], "answer": answer[:20000]})
    return Response(answer, status_code=response.status_code, headers=dict(response.headers))


def stop_server():
    running, thread = globals().get("UVICORN_SERVER"), globals().get("UVICORN_THREAD")
    if running is not None:
        running.should_exit = True
        thread.join(timeout=10)


def stop_agent():
    """Stop the agent (and ngrok). Run stop_agent() in a new cell when you are done."""
    stop_server()
    if USE_NGROK:
        from pyngrok import ngrok
        ngrok.kill()
    print("Stopped")


stop_server()  # an earlier run of this cell: stop that server first
with socket.socket() as probe:
    if probe.connect_ex(("127.0.0.1", PORT)) == 0:
        raise RuntimeError(f"Port {PORT} is already in use (is the agent still running in another window?). "
                           "Stop that program, or set PORT to another number in step 1 and run all cells again.")

UVICORN_SERVER = uvicorn.Server(uvicorn.Config(server.app, host="127.0.0.1", port=PORT, log_level="warning"))
UVICORN_THREAD = threading.Thread(target=UVICORN_SERVER.run, daemon=True)
UVICORN_THREAD.start()
for _ in range(150):
    if UVICORN_SERVER.started:
        break
    if not UVICORN_THREAD.is_alive():
        raise RuntimeError("The agent did not start: see the messages above.")
    time.sleep(0.1)
else:
    raise RuntimeError("The agent did not start within 15 s.")
print("Agent running on", PUBLIC_URL)

In [ ]:
# 7. Test the whole flow through the public address, exactly as Pega will call it
import httpx

claim = json.loads((AGENT_DIR / "examples" / "request_duplicate.json").read_text(encoding="utf-8"))
rpc = {"jsonrpc": "2.0", "id": 1, "method": "message/send",
       "params": {"message": {"role": "user", "messageId": "notebook-test-1", "parts": [{"kind": "data", "data": claim}]}}}
with httpx.Client(base_url=PUBLIC_URL, headers={"ngrok-skip-browser-warning": "1"}, timeout=30) as http:
    card = http.get("/.well-known/agent.json").json()
    token_reply = http.post("/oauth/token", data={"grant_type": "client_credentials", "scope": SCOPE},
                            auth=(CLIENT_ID, CLIENT_SECRET))  # Client ID and Secret sent as "Basic" header
    token_reply.raise_for_status()
    token = token_reply.json()["access_token"]
    in_body = http.post("/oauth/token", data={"grant_type": "client_credentials", "scope": SCOPE,
                                              "client_id": CLIENT_ID, "client_secret": CLIENT_SECRET})
    answer = http.post("/a2a", json=rpc, headers={"Authorization": f"Bearer {token}"}).json()
    as_text = json.loads(json.dumps(rpc))  # the same claim as JSON text, which is how a Pega AI agent sends it
    as_text["params"]["message"].update(messageId="notebook-test-2", parts=[{"kind": "text", "text": json.dumps(claim)}])
    text_answer = http.post("/a2a", json=as_text, headers={"Authorization": f"Bearer {token}"}).json()
    wrong_secret = http.post("/oauth/token", data={"grant_type": "client_credentials"}, auth=(CLIENT_ID, "wrong"))
    no_token = http.post("/a2a", json=rpc)

result = answer["result"]["parts"][0]["data"]
text_result = text_answer["result"]["parts"][0]["data"]
assert card["skills"][0]["id"] == "assess_fraud_risk" and card["url"] == f"{PUBLIC_URL}/a2a", card
assert in_body.status_code == 200, in_body.text
assert (result["status"], result["risk_score"]) == ("COMPLETED", 70), result
assert (text_result["status"], text_result["risk_score"]) == ("COMPLETED", 70), text_result
assert (wrong_secret.status_code, no_token.status_code) == (401, 401)
print("OK  Agent Card found: skill", card["skills"][0]["id"])
print("OK  Token issued for the Client ID and Secret (both 'Basic' and 'in the request body' work)")
print("OK  A2A call with that token:", result["status"], "| risk_score", result["risk_score"], "|", result["risk_flags"])
print("OK  The same call with the claim as JSON text (how a Pega AI agent sends it):", text_result["status"])
print("OK  Wrong secret refused (401), call without a token refused (401)")

In [ ]:
# 8. Enter these values in Pega
from IPython.display import Markdown, display

display(Markdown(f"""
| Pega field | Value |
|---|---|
| **Client ID** | `{CLIENT_ID}` |
| **Client Secret** | `{CLIENT_SECRET}` |
| **Access Token Endpoint** | `{TOKEN_URL}` |
| **Scope** | `{SCOPE}` |
| Grant type | Client credentials |
| Client authentication | Basic, or in the request body (both work) |
| Agent Card URL | `{PUBLIC_URL}/.well-known/agent.json` |
| A2A endpoint | `{PUBLIC_URL}/a2a` (JSON-RPC `message/send`, skill `assess_fraud_risk`) |
| REST fallback (Connect-REST) | `POST {PUBLIC_URL}/v1/assess` |
"""))

## In Pega

1. **Authentication profile**: type *OAuth 2.0*, grant type *Client credentials*. Enter the **Client ID**,
   **Client Secret**, **Access Token Endpoint** and **Scope** from the table above.
2. **Agent connection** (Connect Agent / A2A): use the **Agent Card URL**, choose the skill `assess_fraud_risk`
   and attach the authentication profile. (For Connect-REST, call the REST fallback URL with the same profile.)
3. If a **Pega AI agent** calls this agent (its *External agents* list), tell it in its instructions to send the
   claim as **one JSON object** with the 15 fields; the Agent Card's skill shows a complete example. It may send that
   JSON as plain text: the agent accepts both text and data.
4. Send a test claim. Screen names differ between Pega versions; these are the only values Pega needs.
5. **If Pega shows an error**, run step 9 below: it shows exactly what Pega sent and what the agent answered.

## Good to know

- **Keep this notebook running** while Pega uses the agent. Colab stops idle sessions after a while.
- **The ngrok address changes every run** unless you set `NGROK_DOMAIN` (one free static domain per ngrok account).
  If it changes, update the Access Token Endpoint and Agent Card URL in Pega.
- **The Client Secret** stays the same while this session runs. After a restart a new one is made, unless you paste
  the old one into `CLIENT_SECRET` in step 1.
- **Clear the outputs before sharing this notebook**: step 8 shows the Client Secret.
- **To stop:** run `stop_agent()` in a new cell, or restart the runtime.
- This setup is for **testing with Pega**. In production, tokens come from your identity provider (Keycloak,
  Entra ID or Okta) and the agent runs on a server with HTTPS: see `DEPLOYMENT.md`.

In [ ]:
# 9. If Pega shows an error: run this cell to see exactly what Pega sent and what the agent answered
#    (the first entries are this notebook's own test calls from step 7; the newest call is last)
def show(raw):
    try:
        return json.dumps(json.loads(raw), indent=2, ensure_ascii=False)[:4000]
    except ValueError:
        return raw.decode("utf-8", "replace")[:4000]


if not RECENT_CALLS:
    print("No calls yet. Let Pega call the agent, then run this cell again.")
for call in RECENT_CALLS:
    print(f"=== {call['time']}  POST {call['path']}  ->  HTTP {call['status']}")
    print("--- Pega sent:")
    print(show(call["sent"]))
    print("--- the agent answered:")
    print(show(call["answer"]))
    print()